# Chapter 9 · From Base Model to Assistant

Supervised fine-tuning of the GPT of chapter 7 with conversations, a loss mask, text mixed in so as not to forget the novel, evaluation before and after, and rounds of rewards. PyTorch on the CPU. The whole notebook takes about ten minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Constants, tasks and model

In Colab, the cell downloads the model of chapter 7 and the novel if they are not there.

In [1]:
"""Chapter 9: from base model to assistant. PyTorch on CPU."""
import math
import random
import re
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(".")
MODEL = Path("material/gpt_quixote.pt")
if not MODEL.exists():  # in Colab, download it from the repository
    import urllib.request
    MODEL.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/gpt_quixote.pt", MODEL)
TEXT = Path("material/quixote.txt")
if not TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quixote.txt", TEXT)
SEED = 1337
DEMOS_REVERSE = 300  # written answers for "reverse"
MIX = True           # add text from the novel to each fine-tuning batch
TASKS = {
    "repeat": lambda word: word,
    "uppercase": lambda word: word.upper(),
    "reverse": lambda word: word[::-1],
}


# --- The model of chapters 6 to 8 --------------------------------------------

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["context"]
        size = C // config["heads"]
        self.query = nn.Linear(C, size, bias=False)
        self.key = nn.Linear(C, size, bias=False)
        self.value = nn.Linear(C, size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        return F.softmax(scores, dim=-1) @ v


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.heads = nn.ModuleList(
            [Head(config) for _ in range(config["heads"])])
        self.projection = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norm1, self.norm2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.symbol_embedding = nn.Embedding(vocab_size, C)
        self.position_embedding = nn.Embedding(config["context"], C)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config["layers"])])
        self.norm = nn.LayerNorm(C)
        self.output = nn.Linear(C, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))


def load_model(path=MODEL):
    saved = torch.load(path)
    symbols = saved["symbols"]
    model = GPT(len(symbols), saved["config"])
    model.load_state_dict(saved["model"])
    return model, symbols, {c: i for i, c in enumerate(symbols)}

## 2. Words and conversations

In [2]:
# --- Words and conversations -------------------------------------------------

def prepare(text, to_id):
    """Split the novel and choose words for training and for the test."""
    data = torch.tensor([to_id[c] for c in text], dtype=torch.long)
    a, b = int(0.8 * len(data)), int(0.9 * len(data))
    counts = {}
    for word in re.findall(r"\b[a-z]+\b(?!’)", text[:a]):
        if 3 <= len(word) <= 8 and all(c in to_id for c in word.upper()):
            counts[word] = counts.get(word, 0) + 1
    words = sorted(w for w, n in counts.items() if n >= 3)
    random.Random(SEED).shuffle(words)
    cut = int(0.8 * len(words))
    return data[:a], data[a:b], words[:cut], words[cut:]


def question(task, word):
    return f"User: {task} “{word}”\nAssistant:"


def encode(task, word, to_id, context):
    """Input, targets and mask of a complete conversation."""
    question_text = question(task, word)
    answer = " " + TASKS[task](word) + "\n"
    ids = [to_id[c] for c in question_text + answer]
    mask = [0] * (len(question_text) - 1) + [1] * len(answer)
    padding = context + 1 - len(ids)
    ids += [to_id["\n"]] * padding
    mask += [0] * padding
    return ids[:-1], ids[1:], mask


def conversation_batch(examples, to_id, context, rng, n=24):
    """n conversations, each with a task and a word chosen at random."""
    rows = []
    for _ in range(n):
        task = rng.choice(list(TASKS))
        word = rng.choice(examples[task])
        rows.append(encode(task, word, to_id, context))
    x, y, mask = zip(*rows)
    return (torch.tensor(x), torch.tensor(y),
            torch.tensor(mask, dtype=torch.float))


def text_batch(data, context, generator, n=8):
    maximum = len(data) - context - 1
    starts = torch.randint(maximum, (n,), generator=generator)
    x = torch.stack([data[i:i + context] for i in starts])
    y = torch.stack([data[i + 1:i + context + 1] for i in starts])
    return x, y

In [3]:
torch.set_num_threads(4)
model, symbols, to_id = load_model()
novel, validation, words, test = prepare(TEXT.read_text(encoding="utf-8"), to_id)
print(len(words), "training words and", len(test), "test words:", words[:8])
x, y, mask = encode("reverse", "sancho", to_id, 64)
print("Conversation:", repr("".join(symbols[i] for i in x[:41])))
print("What counts in the loss:", repr("".join(symbols[i] for i, m in zip(y, mask) if m)))

3676 training words and 919 test words: ['galleys', 'mirrors', 'happened', 'backside', 'singer', 'upper', 'fathers', 'repair']
Conversation: 'User: reverse “sancho”\nAssistant: ohcnas\n'
What counts in the loss: ' ohcnas\n'


## 3. Supervised fine-tuning

In [4]:
# --- Supervised fine-tuning --------------------------------------------------

def mean_loss(model, x, y, mask=None):
    logits = model(x)
    losses = F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1),
                             reduction="none")
    if mask is None:
        return losses.mean()
    return (losses * mask.view(-1)).sum() / mask.sum()


def fine_tune(model, optimizer, examples, novel, steps, to_id, rng,
              generator):
    context = model.config["context"]
    for _ in range(steps):
        x, y, mask = conversation_batch(examples, to_id, context, rng)
        loss = mean_loss(model, x, y, mask)
        if MIX:  # a little of the novel at each step, so as not to forget it
            loss = loss + mean_loss(
                model, *text_batch(novel, context, generator))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

## 4. Answering, rewarding and evaluating

In [5]:
# --- Answering and evaluating ------------------------------------------------

@torch.no_grad()
def respond(model, questions, symbols, to_id, temperature=0,
            generator=None):
    """Answer several questions of the same length at once."""
    ids = torch.tensor([[to_id[c] for c in q] for q in questions])
    for _ in range(11):  # space, up to 8 letters, line break and a margin
        logits = model(ids)[:, -1]
        if temperature == 0:
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            probabilities = F.softmax(logits / temperature, dim=-1)
            next_id = torch.multinomial(probabilities, 1,
                                        generator=generator)
        ids = torch.cat([ids, next_id], dim=1)
    n = len(questions[0])
    texts = ["".join(symbols[i] for i in row[n:]) for row in ids.tolist()]
    return [text.split("\n")[0].strip() for text in texts]


def by_length(words):
    groups = {}
    for word in words:
        groups.setdefault(len(word), []).append(word)
    return groups.values()


def reward(task, word, answer):
    """1 if the answer is correct and 0 if not."""
    return int(answer == TASKS[task](word))


def accuracy(model, words, task, symbols, to_id, temperature=0,
             attempts=1, generator=None):
    """Fraction of words with some rewarded answer among the attempts."""
    correct = 0
    for group in by_length(words):
        questions = [question(task, word) for word in group]
        rewarded = set()
        for _ in range(attempts):
            answers = respond(model, questions, symbols, to_id,
                              temperature, generator)
            rewarded |= {word for word, answer
                         in zip(group, answers)
                         if reward(task, word, answer)}
        correct += len(rewarded)
    return correct / len(words)


@torch.no_grad()
def novel_loss(model, data, group=256):
    """Mean loss on a whole block of the novel (chapter 7)."""
    T = model.config["context"]
    n = (len(data) - 1) // T
    x, y = data[:n * T].view(n, T), data[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, group):
        total += mean_loss(model, x[i:i + group],
                           y[i:i + group]).item() * len(x[i:i + group])
    return total / n


def report(model, words, validation, symbols, to_id):
    result = {task: accuracy(model, words, task, symbols, to_id)
              for task in TASKS}
    result["novel"] = novel_loss(model, validation)
    return result


@torch.no_grad()
def continue_text(model, symbols, to_id, start, n, generator):
    """Continue a text, sampling with temperature 0.8 (chapter 8)."""
    context = model.config["context"]
    ids = [to_id[c] for c in start]
    for _ in range(n):
        logits = model(torch.tensor([ids[-context:]]))[0, -1]
        probabilities = F.softmax(logits / 0.8, dim=-1)
        ids.append(int(torch.multinomial(probabilities, 1,
                                         generator=generator)))
    return "".join(symbols[i] for i in ids)

## 5. Learning from its own successes

In [6]:
# --- Rewards: learning from its own successes --------------------------------

def collect_successes(model, words, task, symbols, to_id, generator):
    """The model answers each word once; we keep the rewarded ones."""
    rewarded = []
    for group in by_length(words):
        questions = [question(task, word) for word in group]
        answers = respond(model, questions, symbols, to_id, 1.0,
                          generator)
        rewarded += [word for word, answer in zip(group, answers)
                     if reward(task, word, answer)]
    return rewarded

## 6. The complete experiment

Base model, supervised fine-tuning and three rounds of rewards. It takes about ten minutes.

In [7]:
def show(model, symbols, to_id, word="dulcinea"):
    for task in TASKS:
        questions = [question(task, word)]
        answer = respond(model, questions, symbols, to_id)[0]
        print("    %s “%s” -> %s" % (task, word, answer))


def run(rounds=3):
    torch.set_num_threads(4)
    rng = random.Random(SEED)
    generator = torch.Generator().manual_seed(SEED)
    model, symbols, to_id = load_model()
    text = TEXT.read_text(encoding="utf-8")
    novel, validation, words, test = prepare(text, to_id)
    print("Training and test words:", len(words), len(test))
    results = {"base": report(model, test, validation, symbols, to_id)}
    print("Base model:", results["base"])
    show(model, symbols, to_id)

    start = time.perf_counter()
    examples = {task: words for task in TASKS}
    examples["reverse"] = words[:DEMOS_REVERSE]
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3,
                                  weight_decay=0.1)
    fine_tune(model, optimizer, examples, novel, 1000, to_id, rng, generator)
    results["fine-tuning"] = report(model, test, validation, symbols, to_id)
    print("After supervised fine-tuning:", results["fine-tuning"])
    show(model, symbols, to_id)
    one, best_of_8 = [accuracy(model, test, "reverse", symbols,
                               to_id, 1.0, attempts, generator)
                      for attempts in (1, 8)]
    results["reverse with T=1"] = [one, best_of_8]
    print("Reverse with T=1: one answer %.3f · best of 8 %.3f"
          % (one, best_of_8))

    known = set(examples["reverse"])
    for r in range(1, rounds + 1):
        rewarded = collect_successes(model, words, "reverse",
                                     symbols, to_id, generator)
        new = set(rewarded) - known
        known |= new
        examples["reverse"] = sorted(known)
        fine_tune(model, optimizer, examples, novel, 200, to_id, rng,
                  generator)
        stage = f"round {r}"
        results[stage] = report(model, test, validation, symbols, to_id)
        print("Round", r, "·", len(new), "new rewarded ·",
              len(known), "reverse examples:", results[stage])
    print("Time since fine-tuning: %.0f s" % (time.perf_counter() - start))
    free = continue_text(model, symbols, to_id, "In a village of La Mancha",
                         150, generator)
    print("Free text:", repr(free))
    return results

In [8]:
results = run()

Training and test words: 3676 919
Base model: {'repeat': 0.0, 'uppercase': 0.0, 'reverse': 0.0, 'novel': 1.4797013900732197}
    repeat “dulcinea” -> and the sa
    uppercase “dulcinea” -> and the sa
    reverse “dulcinea” -> and the sa
After supervised fine-tuning: {'repeat': 0.9303590859630033, 'uppercase': 0.9727965179542981, 'reverse': 0.2807399347116431, 'novel': 1.671397068468319}
    repeat “dulcinea” -> dulcines
    uppercase “dulcinea” -> DULCINEA
    reverse “dulcinea” -> enicsuad
Reverse with T=1: one answer 0.173 · best of 8 0.400
Round 1 · 569 new rewarded · 869 reverse examples: {'repeat': 0.9390642002176278, 'uppercase': 0.9630032644178455, 'reverse': 0.500544069640914, 'novel': 1.684684550489178}
Round 2 · 719 new rewarded · 1588 reverse examples: {'repeat': 0.9738846572361263, 'uppercase': 0.984766050054407, 'reverse': 0.6452665941240479, 'novel': 1.6514745472601702}
Round 3 · 685 new rewarded · 2273 reverse examples: {'repeat': 0.9771490750816104, 'uppercase': 0.96300

## Exercises

1. Explain in your own words the three phases of training an assistant. What data does each one use? In which of them is it not necessary to know the correct answer in advance?

2. Write the complete conversation for `uppercase` with the word `sancho`. How many characters does it have, counting the two line breaks? How many predictions count in the loss?

3. Compute the length of the longest conversation that can appear with 8-letter words. Does it fit in the context? What longer word would fit?

4. What else would the model learn if the loss also counted the characters of the question? Would it be a problem in a real assistant?

5. Change `MIX` to `False` and run the program. Compare the loss on the novel, the accuracies and the free text with those of the chapter. What has happened?

6. With a single sampled answer, the model gets 17.3% of the test words right when reversing them. If the eight attempts were independent, what would be the probability of getting at least one right? Compare it with the 40.0% measured. Why is the real value lower?

7. To check that the improvement of the rounds comes from the rewards, replace `collect_successes` with a function that always returns an empty list and run the program. What accuracy does the reverse task reach now? What does the comparison show?

8. Suppose that, instead of keeping the word, we kept the answer the model wrote. Propose two badly designed rewards for the reverse task and explain what the model would learn with each one.

9. Add to `TASKS` a fourth task, `say the first letter of`, whose answer is the first letter of the word, and run the program. (The request must be short: the question and the 11 characters of the answer have to fit in the 64 positions of the context.) Does the model learn it? Does it affect the other three?

10. Report before and after. With the results of your run, write a short report, about ten lines long: what data were used in each phase, how the accuracy in each task and the loss on the novel changed, and what the rewards contributed.

## References

Inspired by [nanochat](https://github.com/karpathy/nanochat), by Andrej Karpathy, and by his video [Deep Dive into LLMs like ChatGPT](https://www.youtube.com/watch?v=7xTGNNLPyMI). Text of *Don Quixote*, by Miguel de Cervantes, in John Ormsby's English translation, in the public domain. The code and the experiments are original.